# E8 — ResNet18-3D nạp trọng số MedicalNet

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Notebook này **tự tải trọng số từ HuggingFace**, xác minh chúng thật sự vào được model,
rồi train. Không cần chuẩn bị Dataset trọng số trước.

⚠️ **Bật `Internet` trong Notebook options** (bảng bên phải) để tải được. Nếu tài khoản
chưa xác minh điện thoại thì Kaggle khoá mục này; khi đó tải file về máy rồi upload
thành Dataset và mount vào, cell tải sẽ tự nhận ra và bỏ qua bước tải.

## Vì sao thử pretrained

316 ca train là rất ít cho một mạng 3D train from scratch. Baseline official của
challenge cũng train from scratch và chỉ đạt 0.6083, tức ngưỡng đó một phần là giới hạn
của **lượng dữ liệu** chứ không chỉ của kiến trúc. MedicalNet được train trên 23 bộ ảnh
y tế 3D nên đặc trưng tầng thấp gần như chắc chắn dùng lại được.

Đây là hướng duy nhất còn lại có thể cho bước nhảy > 0.05. Khoảng cách từ 0.616
(test-104, ensemble E4) tới 0.75 không lấp được bằng tinh chỉnh augmentation: E6, E6b và
E12 đều null.

## ⚠️ E8 đổi HAI thứ cùng lúc

DenseNet121 → ResNet18, **và** from-scratch → pretrained. **E8 thắng thì không quy được
cho pretrained.** Muốn quy kết phải chạy thêm nhánh đối chứng ResNet18 from-scratch, tức
gấp đôi số run. Thứ tự chi quota hợp lý:

1. E8 pretrained, 2 fold. Null thì dừng, khỏi cần đối chứng.
2. Rõ hơn E4 thì chạy đủ 5 fold trước. Bài học E6b: 2 fold cho +0.038, 5 fold cho −0.002.
3. Chỉ khi 5 fold vẫn dương mới chi cho nhánh đối chứng.

## Cách đọc kết quả — chốt TRƯỚC khi chạy

Mốc E4, cùng bệnh nhân, cùng fold, cùng cache:

    fold 1 0.7001 | 2 0.6771 | 3 0.7304 | 4 0.6680 | 5 0.6618 | gộp 394 ca 0.6851

| quan sát trên 5 fold | kết luận |
|---|---|
| CI95 của hiệu không chứa 0, dương | E8 thành cấu hình gốc mới, rồi chạy đối chứng |
| CI95 chứa 0 | giữ E4, ghi lại E8 là một kết quả null |
| epoch `val_loss` chạm đáy muộn hơn | pretrained có chống overfit (ρ=0.770, S-107) |

## ⚠️ Mạng của MONAI KHÔNG giống mạng sinh ra trọng số

Đọc mục này trước khi diễn giải bất kỳ con số nào. File trọng số đến từ Med3D, một mạng
**segmentation**, và nó khác ResNet phân loại của MONAI ở ba chỗ:

| | Med3D (nơi trọng số được học) | MONAI mặc định |
|---|---|---|
| `conv1` | stride (2, 2, 2) | stride (1, 1, 1) |
| `layer3` | stride 1, **dilation 2** | stride 2, dilation 1 |
| `layer4` | stride 1, **dilation 4** | stride 2, dilation 1 |

**Không chỗ nào đổi hình dạng trọng số**, nên cả ba nạp trót lọt và tỉ lệ khớp vẫn ~97%.
Nhưng mọi bộ lọc ở `layer3`/`layer4` được học để nhìn trường tiếp nhận *giãn* ở độ phân
giải cao, còn ở đây chúng nhìn trường đặc ở 1/4 độ phân giải.

`_make_layer` của MONAI không nhận `dilation` nên **không khớp lại được**. Đây là giới
hạn cố hữu của việc dùng trọng số segmentation cho backbone phân loại, và MONAI cũng
chấp nhận đúng như vậy ở đường `pretrained=True` của họ.

**Hệ quả:** E8 null thì *"pretrained không giúp"* không phải lời giải thích duy nhất.
Phải ghi điều này vào báo cáo, không im lặng.

Chỗ duy nhất chỉnh được là `model.conv1_stride` trong config:

| giá trị | bản đồ cuối (vào 112×112×32) | ghi chú |
|---|---|---|
| `1` (mặc định) | 7×7×2 | nhân 7×7×7 ở nguyên độ phân giải, tầng đắt nhất mạng |
| `[1, 2, 2]` | 4×4×2 | hạ mẫu trong mặt phẳng như Med3D, giữ z. Rẻ hơn 4 lần |
| `2` | 4×4×1 | khớp Med3D hẳn, nhưng z còn 1 lát — **đừng dùng** với hình học này |

## Lỗi đã sửa trong lần chuẩn bị này

`configs/e8_pretrained.yaml` từng để `shortcut_type: B`, `bias_downsample: false` cho
resnet18. **Sai**: file trọng số MedicalNet resnet18/34 sinh từ biến thể `("A", true)`.

Điều khiến lỗi này nguy hiểm là nó **không tự lộ ra**. Shortcut "A" là avg-pool cộng đệm
0 và không có tham số nào; "B" dựng thêm conv 1×1 + norm ở ba chỗ nối tầng. Đặt sai thì
~18 khoá nằm trên đường tắt của 3/4 stage khởi tạo ngẫu nhiên, mà tỉ lệ khớp vẫn báo
~85% — dư sức qua ngưỡng 50% của cổng cũ. Cổng nay dựa trên **khoá nào** thiếu, không
phải **bao nhiêu** khoá thiếu.

## 0. Bootstrap

Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào. Phải là `84453a2` trở về
sau, vì bản trước đó còn mang lỗi `shortcut_type`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2]                     # sàng 2 fold trước; đủ 5 fold mới kết luận
CONFIG_NAME = "e8_pretrained.yaml"
SCOPE = "model."                   # khối config thí nghiệm này ĐƯỢC PHÉP đổi
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

EXPERIMENT = Path(CONFIG_NAME).stem
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
CFG = load_yaml(CFG_PATH)

print(f"\nthí nghiệm: {EXPERIMENT} · fold {FOLDS}")
print(f"model: {CFG['model']['name']} depth {CFG['model']['depth']} · "
      f"shortcut {CFG['model']['shortcut_type']} · bias_downsample "
      f"{CFG['model']['bias_downsample']}")

## Cổng 0 ⚠️ — config này khác baseline ở ĐÚNG những chỗ nào

E8 đổi kiến trúc nên khác baseline nhiều khoá hơn các thí nghiệm trước. Điều bắt buộc là
mọi khác biệt phải nằm **trong khối `model:`**. Một khoá lọt ra ngoài, ví dụ `data.augment`
hay `train.lr`, biến phép so kiến trúc thành phép so hai biến, và sai đó không để lại
dấu vết nào trong kết quả.

In [ ]:
BASE = load_yaml(REPO / "configs" / "baseline_3dpatch.yaml")


def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flatten(v, prefix + k + "."))
        else:
            out[prefix + k] = v
    return out


fa, fb = flatten(BASE), flatten(CFG)
diff = {
    k: (fa.get(k), fb.get(k))
    for k in sorted(set(fa) | set(fb))
    if str(fa.get(k)) != str(fb.get(k))
}

# `output_dir` và `fold` phân biệt lần chạy, không phải biến của thí nghiệm.
MIEN = ("output_dir", "fold")

print(f"{CONFIG_NAME} khác baseline ở {len(diff)} khoá:")
for k, (a, b) in diff.items():
    ghi_chu = "   (miễn)" if k in MIEN else ("" if k.startswith(SCOPE) else "   <-- NGOÀI")
    print(f"  {k}: {a!r} -> {b!r}{ghi_chu}")

ngoai = [k for k in diff if not k.startswith(SCOPE) and k not in MIEN]
assert not ngoai, f"⛔ khác biệt NGOÀI {SCOPE}: {ngoai} — không còn là phép so kiến trúc"
print(f"\n✓ chỉ khác trong {SCOPE} (+ {', '.join(MIEN)}) — so sánh có kiểm soát")

## 1. Trọng số MedicalNet

Ba nguồn, thử theo thứ tự. Nguồn nào có trước thì dùng, không tải lại:

1. **Đã mount** thành Kaggle Dataset — nhanh nhất, không cần internet.
2. **Đã tải trong session này** (`/kaggle/working/weights`).
3. **Tải từ HuggingFace** `TencentMedicalNet/MedicalNet-Resnet18`, file
   `resnet_18_23dataset.pth`, 132 MB.

File tải về nằm trong `/kaggle/working/weights`, nên **Save Version** sẽ giữ lại và bạn
mount được cho các session sau, khỏi phụ thuộc internet.

⚠️ Có hai file trong repo đó: `resnet_18.pth` và `resnet_18_23dataset.pth`. Bản
`_23dataset` là bản train trên đủ 23 bộ dữ liệu, là bản dự án chọn.

In [ ]:
MEDICALNET_FILE = "resnet_18_23dataset.pth"
HF_REPO = "TencentMedicalNet/MedicalNet-Resnet18"
MIN_BYTES = 100 * 1024**2       # file thật 132 MB; nhỏ hơn nhiều = tải hỏng hoặc HTML lỗi

WEIGHTS_DIR = Path("/kaggle/working/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
dest = WEIGHTS_DIR / MEDICALNET_FILE

mounted = [p for p in sorted(Path("/kaggle/input").rglob("resnet_18*.pth"))]
if mounted:
    WEIGHTS = mounted[0]
    print(f"dùng bản đã mount: {WEIGHTS}")
elif dest.exists() and dest.stat().st_size >= MIN_BYTES:
    WEIGHTS = dest
    print(f"dùng bản đã tải trong session này: {WEIGHTS}")
else:
    print(f"tải từ HuggingFace: {HF_REPO}/{MEDICALNET_FILE} ...")
    url = f"https://huggingface.co/{HF_REPO}/resolve/main/{MEDICALNET_FILE}"
    try:
        from huggingface_hub import hf_hub_download

        WEIGHTS = Path(hf_hub_download(
            repo_id=HF_REPO, filename=MEDICALNET_FILE, local_dir=str(WEIGHTS_DIR)
        ))
    except Exception as exc:  # noqa: BLE001 - hạ cấp sang urllib rồi báo lỗi tử tế
        print(f"  huggingface_hub không dùng được ({type(exc).__name__}), thử urllib")
        import urllib.error
        import urllib.request

        try:
            urllib.request.urlretrieve(url, dest)
            WEIGHTS = dest
        except (urllib.error.URLError, OSError) as exc2:
            raise RuntimeError(
                f"Không tải được trọng số ({exc2}).\n"
                "  Gần như chắc chắn Internet đang TẮT. Bật ở Notebook options (bảng\n"
                "  bên phải) rồi chạy lại cell này. Nếu tài khoản chưa xác minh điện\n"
                f"  thoại thì Kaggle khoá mục đó — khi đó tải tay từ\n    {url}\n"
                "  rồi upload thành Kaggle Dataset và mount vào; cell này sẽ tự nhận."
            ) from None

size = WEIGHTS.stat().st_size
assert size >= MIN_BYTES, (
    f"{WEIGHTS} chỉ {size / 1e6:.1f} MB, cần khoảng 132 MB. Nhiều khả năng đây là trang "
    "lỗi HTML được lưu thành .pth. Xoá file và chạy lại."
)

os.environ["LLDMMRI_PRETRAINED_PATH"] = str(WEIGHTS)
print(f"\ntrọng số: {WEIGHTS}  ({size / 1e6:.0f} MB)")

## Cổng A ⚠️⚠️ — trọng số có THẬT SỰ vào model không

Cổng quan trọng nhất của notebook. Chế độ hỏng của pretrained không phải crash mà là
**im lặng**: model vẫn dựng, vẫn train, vẫn ra số, chỉ là một phần trọng số ngẫu nhiên.
Cả thí nghiệm "có pretrained" biến thành "không pretrained" mà không ai biết.

Ba phép kiểm, mỗi phép chặn một chế độ hỏng khác nhau:

1. **`build_model` không nổ** — cặp `shortcut_type`/`bias_downsample` khớp biến thể sinh
   ra file trọng số, và không khoá nào thiếu ngoài đầu phân loại.
2. **Trùng khớp bit-exact với file** — chứng minh giá trị trong model đúng là giá trị
   trong file, không phải "không báo lỗi" nên cho là xong.
3. **`conv1` đã chia cho số kênh** — trọng số gốc là 1 kênh, ta có 8 thì. Nhân bản mà
   không chia thì tiền kích hoạt lớn gấp 8, và mọi BatchNorm phía sau đã học thống kê
   cho thang cũ. Sai thang làm hỏng đúng thứ khiến pretrained có giá trị.

In [ ]:
import torch

from src.models import build_model, count_parameters

# build_model in báo cáo khớp khoá, và NỔ nếu sai biến thể hoặc thiếu khoá ngoài fc.*
model = build_model(CFG["model"])
print(f"tham số: {count_parameters(model):,}")

raw = torch.load(WEIGHTS, map_location="cpu")
state = raw.get("state_dict", raw) if isinstance(raw, dict) else raw
state = {k.replace("module.", "", 1): v for k, v in state.items()}
sd = model.state_dict()

bit_exact = [
    k for k, v in state.items()
    if k in sd and v.shape == sd[k].shape and torch.equal(v, sd[k])
]
print(f"khoá trùng khớp bit-exact với file: {len(bit_exact)}/{len(sd)}")
assert len(bit_exact) > 0.7 * len(sd), (
    "phần lớn trọng số KHÔNG đến từ file — model gần như ngẫu nhiên đội lốt pretrained"
)

w, src = sd["conv1.weight"], state["conv1.weight"]
assert w.shape[1] == CFG["model"]["in_channels"], w.shape
assert src.shape[1] == 1, src.shape
assert torch.allclose(w[:, 0], src[:, 0] / float(w.shape[1]), atol=1e-6), (
    "conv1 chưa được chia cho số kênh — thang kích hoạt lệch 8 lần"
)
print(f"conv1: 1 kênh -> {w.shape[1]} kênh, đã chia cho {w.shape[1]} ✓")

thieu = [k for k in sd if k not in state]
print(f"khoá thiếu (phải chỉ là đầu phân loại): {thieu}")

## 2. Cache E4

E8 dùng lại **đúng cache của E4** để phép so chỉ khác ở khối `model:`.

Nhận diện bằng **nội dung** `cache_meta.json`, không bằng tên dataset: tên do người
upload đặt và đã lệch một lần (S-080).

⚠️ **Phải loại cache E12.** Nó cũng `per_phase` + `lesion_tight` + `target_size`
112×112×32 nên ba khoá thường dùng không phân biệt được. Khác biệt là `crop_margin_voxels`:
E12 có lề dư và mảng thật là 136×136×40. Cho cache E12 vào config này thì model nhận khối
136×136×40, hình học khác hẳn E4, và **không có gì báo lỗi**.

In [ ]:
import json as _json

E4_KEYS = {
    "align_phases": "per_phase",
    "crop_mode": "lesion_tight",
    "target_size": [112, 112, 32],
}
GRID = (8, 112, 112, 32)

ung_vien, CACHE_DIR = [], None
for meta_path in sorted(Path("/kaggle/input").rglob("cache_meta.json")):
    try:
        meta = _json.loads(meta_path.read_text("utf-8"))
    except Exception:  # noqa: BLE001 - chỉ để liệt kê chẩn đoán
        continue
    le = meta.get("crop_margin_voxels")
    khop = all(meta.get(k) == v for k, v in E4_KEYS.items()) and not any(le or [])
    ung_vien.append((meta_path.parent, meta, khop))
    if khop and CACHE_DIR is None:
        CACHE_DIR = meta_path.parent

print(f"=== {len(ung_vien)} cache tìm thấy dưới /kaggle/input ===")
for path, meta, khop in ung_vien:
    print(f"  {'✓ E4 ' if khop else '  -- '}  {path}")
    print(f"          size={meta.get('target_size')} lề={meta.get('crop_margin_voxels')} "
          f"align={meta.get('align_phases')} crop={meta.get('crop_mode')}")

if CACHE_DIR is None:
    raise RuntimeError(
        "Chưa mount cache E4.\n"
        f"  Cần cache có {E4_KEYS} và KHÔNG có crop_margin_voxels.\n"
        "  Cache E12 (lề 12/12/4, mảng 136x136x40) KHÔNG dùng được cho config này."
    )

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
print(f"\ncache: {CACHE_DIR}")

## Cổng B ⚠️ — hình dạng mảng thật đúng chưa

`cache_meta.json` là thứ người viết cache khai; mảng `.npz` là thứ model thật sự nhận.
Kiểm cả hai.

In [ ]:
import numpy as np

meta = _json.loads((CACHE_DIR / "cache_meta.json").read_text("utf-8"))
for k, v in E4_KEYS.items():
    assert meta.get(k) == v, f"cache SAI: {k} = {meta.get(k)!r}, cần {v!r}"

n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498"

mau = next(CACHE_DIR.glob("*.npz"))
with np.load(mau) as z:
    shape = tuple(z["image"].shape)
assert shape == GRID, f"hình dạng {shape}, cần {GRID} — đây không phải cache E4"
print(f"cache E4 ✓ · {n_npz} ca · mảng {shape}")
print(f"commit build: {meta.get('git_commit')}")

## 3. Ngân sách — E8 có thể đổi hẳn nút thắt

Với E4 (DenseNet121) thì GPU chỉ tốn ~20 s/epoch còn tổng là 45 s/epoch: **CPU chặn**,
GPU ngồi chờ dữ liệu.

E8 có lý do để khác. MONAI dựng `conv1` với stride **(1,1,1)**, tức tầng đầu chạy nhân
7×7×7 ở nguyên 112×112×32 trước khi có bất kỳ phép hạ mẫu nào. Riêng tầng đó nặng hơn
toàn bộ phần thân mạng. Cell này đo GPU thuần trước khi chi 4 giờ.

Đọc kết quả:

| GPU đo được | nghĩa là |
|---|---|
| dưới ~40 s/epoch | vẫn CPU chặn như E4, ngân sách tương đương, chạy tiếp |
| 40–70 s/epoch | GPU thành nút thắt, mỗi fold đắt hơn E4 |
| trên ~70 s/epoch | đặt `model.conv1_stride: [1, 2, 2]` trong config rồi chạy lại từ bootstrap |

`[1, 2, 2]` hạ mẫu trong mặt phẳng như Med3D và **giữ nguyên trục z**, rẻ hơn 4 lần. Đừng
dùng `2`: z chỉ có 32 voxel, hạ mẫu 32 lần thì block cuối còn đúng một lát.

In [ ]:
import time

from src.train.run import build_loaders

train_loader, val_loader, _ = build_loaders(CFG, FOLDS[0])
so_batch = len(train_loader)

dev = torch.device("cuda")
model = model.to(dev).train()
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda")

bs = int(CFG["data"]["batch_size"])
x = torch.randn(bs, CFG["model"]["in_channels"], 112, 112, 32, device=dev)
y = torch.zeros(bs, dtype=torch.long, device=dev)


def buoc():
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16):
        out = model(x)
        loss = torch.nn.functional.cross_entropy(out, y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    return out


for _ in range(3):
    out = buoc()
assert tuple(out.shape) == (bs, CFG["model"]["num_classes"]), out.shape

torch.cuda.synchronize()
t0 = time.time()
for _ in range(15):
    buoc()
torch.cuda.synchronize()
giay_batch = (time.time() - t0) / 15

# CPU thuần: nạp + augment, không đụng GPU
t0, n = time.time(), 0
for _ in train_loader:
    n += 1
    if n >= 20:
        break
cpu_epoch = (time.time() - t0) / n * so_batch
gpu_epoch = giay_batch * so_batch

print(f"shape ra: {tuple(out.shape)} · VRAM đỉnh "
      f"{torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
print(f"GPU fwd+bwd : {gpu_epoch:6.0f} s/epoch  ({so_batch} batch × {giay_batch:.3f}s)")
print(f"CPU nạp+aug : {cpu_epoch:6.0f} s/epoch")
print(f"ước tính    : {max(cpu_epoch, gpu_epoch):6.0f} s/epoch -> "
      f"{max(cpu_epoch, gpu_epoch) * int(CFG['train']['epochs']) / 3600:.1f} h/fold")
print(f"{len(FOLDS)} fold -> "
      f"{max(cpu_epoch, gpu_epoch) * int(CFG['train']['epochs']) * len(FOLDS) / 3600:.1f} h")
print("\n(E4 DenseNet: GPU ~20 s/epoch, tổng 45 s/epoch, 3.76 h/fold)")

del model, opt, x, y
torch.cuda.empty_cache()

## 4. Train

`resume: true` nên bị ngắt giữa chừng thì chạy lại đúng cell này, nó đọc tiếp từ `last.pt`.

⚠️ `train` đọc YAML **từ đĩa**, không dùng biến `CFG` trong notebook. Sửa `CFG` bằng tay
ở cell nào đó sẽ không có tác dụng lúc train, mà cổng 0 lại đọc từ chính file nên vẫn báo
xanh. Muốn đổi tham số thì sửa file config rồi chạy lại cell bootstrap.

Đường dẫn trọng số đi qua env `LLDMMRI_PRETRAINED_PATH` nên `train` thấy được nó.

In [ ]:
import time

from src.train.run import train

assert os.environ.get("LLDMMRI_PRETRAINED_PATH"), "env trọng số trống — chạy lại mục 1"

t0 = time.time()
results = {}

for fold in FOLDS:
    da_dung = (time.time() - t0) / 3600
    print("\n" + "=" * 60)
    print("FOLD %d  (da dung %.2fh)" % (fold, da_dung))
    print("=" * 60)
    results[fold] = train(CFG_PATH, fold_override=fold)
    print("fold %d xong: macro-F1 %.4f" % (fold, results[fold]["macro_f1"]))

print("\ntong: %.2f h" % ((time.time() - t0) / 3600))

## 5. Kết quả session này

⚠️ Chênh lệch từng fold là **nhiễu nếu nhìn riêng** (CI mỗi fold khoảng ±0.19). Bảng này
để phát hiện bất thường, không để kết luận. Kết luận cần đủ 5 fold và so cặp trên 394 ca
chạy ở local.

In [ ]:
import csv as _csv

OUT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
E4 = {1: 0.7001, 2: 0.6771, 3: 0.7304, 4: 0.6680, 5: 0.6618}
E4_DAY = {1: 100, 2: 79, 3: 227, 4: 3, 5: 14}

print(f"{'fold':>5}{'epoch':>7}{'macro-F1':>11}{'kappa':>9}{'E4':>9}{'hiệu':>9}")
print("-" * 50)
rows = []
for d in sorted(OUT.glob("fold*")):
    f = d / "metrics_best.json"
    if not f.exists():
        continue
    m = _json.loads(f.read_text("utf-8"))
    fold = int(m["fold"])
    hieu = m["macro_f1"] - E4[fold]
    rows.append((d, fold))
    print(f"{fold:>5}{m['epoch']:>7}{m['macro_f1']:>11.4f}"
          f"{m['cohen_kappa']:>9.4f}{E4[fold]:>9.4f}{hieu:>+9.4f}")

print("\nEpoch chạm đáy val_loss — dự báo gần trọn vẹn F1 cuối (ρ=0.770, S-107):")
for d, fold in rows:
    log = d / "train_log.csv"
    if log.exists():
        vl = [float(x["val_loss"]) for x in _csv.DictReader(open(log))]
        print(f"  fold {fold}: đáy @ epoch {vl.index(min(vl)) + 1:>3}   (E4: {E4_DAY[fold]})")
print("  Muộn hơn = overfit muộn hơn. Đây là tín hiệu độc lập với macro-F1.")

## 6. Gói mang về

Hai thứ: kết quả train, và **file trọng số** nếu vừa tải. Giữ lại thì session sau mount
được thay vì phụ thuộc internet.

In [ ]:
import shutil

PACK = Path(f"/kaggle/working/{EXPERIMENT}_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

KEEP = ["val_probs_best.npz", "val_probs_last.npz", "metrics_best.json",
        "train_log.csv", "config_used.json", "best.pt"]
for d in sorted(OUT.glob("fold*")):
    dst = PACK / d.name
    dst.mkdir(parents=True, exist_ok=True)
    for name in KEEP:
        if (d / name).exists():
            shutil.copy2(d / name, dst / name)

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total / 2**20:.1f} MiB")
if WEIGHTS.is_relative_to(Path("/kaggle/working")):
    print(f"trọng số giữ ở {WEIGHTS.parent} — Save Version rồi mount cho session sau")

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz và .pt bản thân là zip; trình giải nén bung
  đệ quy sẽ biến chúng thành thư mục và src.eval.* không thấy (đã dính, S-078).

Ở local, sau khi đủ 5 fold:
    python -m src.eval.run --run-dir runs/e8_pretrained
""")